In [6]:
import os
import pandas as pd
from sqlalchemy import create_engine
import numpy as np
from scipy.stats import ks_2samp

SAMPLE_SIZE = 10_000  # seguro para la memoria disponible

db_url = (
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:"
    f"{os.environ['POSTGRES_PASSWORD']}@postgres:5432/"
    f"{os.environ['POSTGRES_DB']}"
)
engine = create_engine(db_url)

In [4]:

batches = pd.read_sql("""
    SELECT
        batch_id,
        COUNT(*) AS registros,
        MIN(target_price) AS precio_minimo,
        AVG(target_price) AS precio_promedio,
        MAX(target_price) AS precio_maximo
    FROM gold.property_training_features
    GROUP BY batch_id
    ORDER BY batch_id
""", engine)

batches

,batch_id,registros,precio_minimo,precio_promedio,precio_maximo
0,group-5-batch-4,320696,300023.0,3.973394e+05,500000.0
1,group-5-batch-5,361457,500007.0,1.134678e+06,515000000.0


In [5]:
REFERENCE_BATCH = "group-5-batch-4"
CURRENT_BATCH = "group-5-batch-5"
SAMPLE_SIZE = 10_000

reference_df = pd.read_sql(
    """
    SELECT *
    FROM gold.property_training_features
    WHERE batch_id = %(batch_id)s
    LIMIT %(limit)s
    """,
    engine,
    params={"batch_id": REFERENCE_BATCH, "limit": SAMPLE_SIZE},
)

current_df = pd.read_sql(
    """
    SELECT *
    FROM gold.property_training_features
    WHERE batch_id = %(batch_id)s
    LIMIT %(limit)s
    """,
    engine,
    params={"batch_id": CURRENT_BATCH, "limit": SAMPLE_SIZE},
)

print("Referencia:", reference_df.shape)
print("Lote nuevo:", current_df.shape)

Referencia: (10000, 15)
Lote nuevo: (10000, 15)


In [7]:
NUMERIC_FEATURES = [
    "brokered_by", "bed", "bath", "acre_lot",
    "street", "zip_code", "house_size", "prev_sold_year",
]

def population_stability_index(reference, current, bins=10):
    reference = reference.dropna()
    current = current.dropna()

    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return np.nan

    edges[0], edges[-1] = -np.inf, np.inf
    intervals = pd.cut(reference, bins=edges, include_lowest=True).cat.categories

    ref_share = (
        pd.cut(reference, bins=edges, include_lowest=True)
        .value_counts()
        .reindex(intervals, fill_value=0) / len(reference)
    )
    cur_share = (
        pd.cut(current, bins=edges, include_lowest=True)
        .value_counts()
        .reindex(intervals, fill_value=0) / len(current)
    )

    epsilon = 1e-6
    return float(
        ((cur_share - ref_share) *
         np.log((cur_share + epsilon) / (ref_share + epsilon))).sum()
    )

numeric_drift = []

for feature in NUMERIC_FEATURES:
    if feature not in reference_df.columns:
        continue

    reference = reference_df[feature]
    current = current_df[feature]

    ks_statistic, ks_pvalue = ks_2samp(
        reference.dropna(),
        current.dropna(),
    )
    psi = population_stability_index(reference, current)

    numeric_drift.append({
        "feature": feature,
        "psi": round(psi, 4),
        "ks_statistic": round(float(ks_statistic), 4),
        "ks_pvalue": float(ks_pvalue),
        "drift_detected": bool(psi >= 0.25 or ks_statistic >= 0.10),
    })

numeric_drift_df = pd.DataFrame(numeric_drift).sort_values(
    ["drift_detected", "psi"],
    ascending=[False, False],
)

numeric_drift_df

,feature,psi,ks_statistic,ks_pvalue,drift_detected
5,zip_code,12.4056,1.0000,0.000000e+00,True
3,acre_lot,0.6621,0.2249,5.699123e-222,True
2,bath,0.6168,0.2653,8.952776e-310,True
7,prev_sold_year,0.5665,0.3291,9.881313e-324,True
6,house_size,0.4323,0.2304,4.864387e-233,True
1,bed,0.3017,0.2105,2.672649e-194,True
0,brokered_by,0.1455,0.0939,8.991347e-39,False
4,street,0.0191,0.0285,5.931599e-04,False


In [8]:
CATEGORICAL_FEATURES = ["zip_code", "status", "city", "state"]

def categorical_metrics(reference, current):
    reference = reference.fillna("__MISSING__").astype(str)
    current = current.fillna("__MISSING__").astype(str)

    ref_share = reference.value_counts(normalize=True)
    cur_share = current.value_counts(normalize=True)
    categories = ref_share.index.union(cur_share.index)

    ref_share = ref_share.reindex(categories, fill_value=0)
    cur_share = cur_share.reindex(categories, fill_value=0)

    epsilon = 1e-6
    psi = float(
        ((cur_share - ref_share) *
         np.log((cur_share + epsilon) / (ref_share + epsilon))).sum()
    )

    total_variation_distance = float(
        0.5 * (ref_share - cur_share).abs().sum()
    )

    new_category_rate = float(
        (~current.isin(set(reference))).mean()
    )

    return psi, total_variation_distance, new_category_rate

categorical_drift = []

for feature in CATEGORICAL_FEATURES:
    psi, tvd, new_rate = categorical_metrics(
        reference_df[feature],
        current_df[feature],
    )

    categorical_drift.append({
        "feature": feature,
        "psi": round(psi, 4),
        "distribution_change": round(tvd, 4),
        "new_category_rate": round(new_rate, 4),
        "drift_detected": bool(psi >= 0.25 or tvd >= 0.20),
    })

categorical_drift_df = pd.DataFrame(categorical_drift).sort_values(
    "distribution_change",
    ascending=False,
)

categorical_drift_df

,feature,psi,distribution_change,new_category_rate,drift_detected
0,zip_code,14.5109,1.0000,1.0000,True
3,state,24.2449,1.0000,1.0000,True
2,city,15.1535,0.9731,0.9222,True
1,status,0.0000,0.0000,0.0000,False
